In [23]:
import pandas as pd
import jellyfish

In [24]:
train_df = pd.read_parquet("./train.parquet")
val_df = pd.read_parquet("./val.parquet")
test_df = pd.read_parquet("./test.parquet")


In [25]:
print ("Our desired 1:0 ration should be 1:9 to better represent the true population")
for name, df in zip (["train","val","test"], [train_df, val_df, test_df]):
    print(f"{'='*10}{name}{'='*30}")
    distr = df["label"].value_counts()
    display(distr.head())
    total = distr[0] + distr[1]
    print(f"1:0 ratio {distr[1]/total:.3f}:,{distr[0]/total:.3f}")


Our desired 1:0 ration should be 1:9 to better represent the true population
==========train==============================


label
0    13592
1     1510
Name: count, dtype: int64

1:0 ratio 0.100:,0.900
==========val==============================


label
0    789
1     87
Name: count, dtype: int64

1:0 ratio 0.099:,0.901
==========test==============================


label
0    677
1     75
Name: count, dtype: int64

1:0 ratio 0.100:,0.900


In [52]:
from sklearn.linear_model import LogisticRegression
import textdistance

def _evaluation_function(s):
    #character-based
    jw_street_sim = jellyfish.jaro_winkler_similarity(s["StreetName_a"], s["StreetName_b"])
    jw_zipcode_sim =  jellyfish.jaro_winkler_similarity(s["ZipCode_a"], s["ZipCode_b"])
    jw_address_sim =  jellyfish.jaro_winkler_similarity(str(s["Address_a"]), str(s["Address_b"])) 
    lv_street_sim = 1 - jellyfish.levenshtein_distance(s["StreetName_a"], s["StreetName_b"])/(max(len(s["StreetName_a"]), len(s["StreetName_b"])))
    lv_zipcode_sim = 1 - jellyfish.levenshtein_distance(s["ZipCode_a"], s["ZipCode_b"])/max((len(s["ZipCode_a"]), len(s["ZipCode_b"])))
    lv_address_sim = 1 - jellyfish.levenshtein_distance(str(s["Address_a"]), str(s["Address_b"]))/max(len(str(s["Address_a"])), len(str(s["Address_b"])))
    hm_zipcode_sim = 1 - jellyfish.hamming_distance(s["ZipCode_a"], s["ZipCode_b"])/5

    #n-grams
    metrics = {
        "jaccard": (textdistance.Jaccard(qval = 2), textdistance.Jaccard(qval = 1)),
        "sorenson":(textdistance.Sorensen(qval = 2), textdistance.Sorensen(qval = 1)),
        "overlap": (textdistance.Overlap(qval=2), textdistance.Overlap(qval=1)),
        "tversky": (textdistance.Tversky(qval=2), textdistance.Tversky(qval=1)),
        "cosine": (textdistance.Cosine(qval = 2),textdistance.Cosine(qval = 1) )
    }

    n_gram_results = {}
    for option, (initial, for_zip) in metrics.items():
        street_score = initial.normalized_similarity(s["StreetName_a"], s["StreetName_b"])
        zip_score = initial.normalized_similarity(s["ZipCode_a"], s["ZipCode_b"])
        address_score = for_zip.normalized_similarity(str(s["Address_a"]), str(s["Address_b"]))
        n_gram_results[f"{option}_street"] = street_score
        n_gram_results[f"{option}_zip"] = zip_score
        n_gram_results[f"{option}_address"] = address_score

    return pd.Series(
        {
            **n_gram_results,
            "jw_street_sim": jw_street_sim,
            "jw_zipcode_sim": jw_zipcode_sim,
            "jw_address_sim": jw_address_sim,
            "lv_street_sim": lv_street_sim,
            "lv_zipcode_sim": lv_zipcode_sim,
            "lv_address_sim": lv_address_sim,
            "hm_zipcode_sim": hm_zipcode_sim,
        })
def _process_df(df):
    return pd.DataFrame([_evaluation_function(row) for _,row in df.iterrows()])
model = LogisticRegression()
X_train = _process_df(train_df)
model.fit(X_train.values, train_df["label"])

preds = model.predict(_process_df(test_df))
probs = model.predict_proba(_process_df(test_df))
probs
model.classes_
print(pd.Series(model.coef_[0], index = X_train.columns))

from sklearn.metrics import classification_report
print(classification_report(test_df["label"], preds))

/Users/david/Downloads/ml-geocoding/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


jaccard_street      -1.000758
jaccard_zip          2.923659
jaccard_address      3.149965
sorenson_street      1.044739
sorenson_zip         1.927694
sorenson_address     1.203517
overlap_street       0.556610
overlap_zip          1.927694
overlap_address      0.629675
tversky_street      -1.000758
tversky_zip          2.923659
tversky_address      3.149965
cosine_street        0.560173
cosine_zip           1.927694
cosine_address       1.140783
jw_street_sim        0.327076
jw_zipcode_sim       1.000927
jw_address_sim       1.742210
lv_street_sim       10.084977
lv_zipcode_sim       1.693456
lv_address_sim       3.584151
hm_zipcode_sim       1.693456
dtype: float64
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       677
           1       0.86      0.91      0.88        75

    accuracy                           0.98       752
   macro avg       0.93      0.95      0.93       752
weighted avg       0.98      0.98      0.98       752

/Users/david/Downloads/ml-geocoding/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
